# 07 - Ranking Analysis

This notebook identifies top and bottom performers across products, customers, and categories.

Focus areas:
- Top products by revenue
- Bottom products by revenue
- Top customers by revenue
- Customer ranking within each country
- Product ranking within each category
- Revenue contribution and cumulative contribution

In [0]:
%sql
/*
Top/Bottom Products by Revenue

Purpose:
    Identify the highest/lowest revenue-generating products.
    Revenue is calculated from the sales fact table and enriched with product attributes.
*/

SELECT
    p.product_name,
    p.category,
    p.subcategory,
    SUM(f.sales_amount) AS total_revenue,
    SUM(f.quantity) AS total_quantity,
    COUNT(DISTINCT f.order_number) AS total_orders
FROM datawarehouseanalytics_gold.fact_sales f
LEFT JOIN datawarehouseanalytics_gold.dim_products p
    ON f.product_key = p.product_key
GROUP BY
    p.product_name,
    p.category,
    p.subcategory
ORDER BY total_revenue DESC --ASC
LIMIT 10;

In [0]:
%sql
/*
Product Revenue Ranking

Purpose:
    Rank products by total revenue using DENSE_RANK.
    DENSE_RANK keeps equal revenue values at the same rank without creating gaps.
*/

WITH product_sales AS (
    SELECT
        p.product_name,
        COALESCE(p.category, 'Unknown') AS category,
        COALESCE(p.subcategory, 'Unknown') AS subcategory,
        SUM(f.sales_amount) AS total_revenue,
        SUM(f.quantity) AS total_quantity,
        COUNT(DISTINCT f.order_number) AS total_orders
    FROM datawarehouseanalytics_gold.fact_sales f
    LEFT JOIN datawarehouseanalytics_gold.dim_products p
        ON f.product_key = p.product_key
    GROUP BY
        p.product_name,
        COALESCE(p.category, 'Unknown'),
        COALESCE(p.subcategory, 'Unknown')
)

SELECT
    product_name,
    category,
    subcategory,
    total_revenue,
    total_quantity,
    total_orders,
    DENSE_RANK() OVER (ORDER BY total_revenue DESC) AS revenue_rank
FROM product_sales
ORDER BY revenue_rank
LIMIT 10;

In [0]:
%sql
/*
Top Products Within Each Category

Purpose:
    Rank products within their own category instead of ranking all products together.
    This gives a fairer comparison between products from the same business group.
*/
WITH product_sales AS (
    SELECT
        COALESCE(p.category, 'Unknown') AS category,
        p.product_name,
        SUM(f.sales_amount) AS total_revenue
    FROM datawarehouseanalytics_gold.fact_sales f
    LEFT JOIN datawarehouseanalytics_gold.dim_products p
        ON f.product_key = p.product_key
    GROUP BY
        COALESCE(p.category, 'Unknown'),
        p.product_name
),

ranked_products AS (
    SELECT
        category,
        product_name,
        total_revenue,
        DENSE_RANK() OVER (
            PARTITION BY category
            ORDER BY total_revenue DESC
        ) AS category_rank
    FROM product_sales
)

SELECT *
FROM ranked_products
WHERE category_rank <= 3
ORDER BY category, category_rank;


In [0]:
%sql
/*
Top Customers by Revenue

Purpose:
    Identify customers who generated the highest total revenue.
    This helps highlight high-value customers.
*/

SELECT
    c.customer_key,
    c.customer_number,
    c.first_name,
    c.last_name,
    c.country,
    COUNT(DISTINCT f.order_number) AS total_orders,
    SUM(f.sales_amount) AS total_revenue,
    ROUND(AVG(f.sales_amount), 2) AS avg_sales_per_line
FROM datawarehouseanalytics_gold.fact_sales f
LEFT JOIN datawarehouseanalytics_gold.dim_customers c
    ON f.customer_key = c.customer_key
GROUP BY
    c.customer_key,
    c.customer_number,
    c.first_name,
    c.last_name,
    c.country
ORDER BY total_revenue DESC
LIMIT 10;

In [0]:
%sql
/*
Top Customers Within Each Country

Purpose:
    Rank customers within each country by revenue.
    This shows the strongest customers per market.
*/

WITH customer_sales AS (
    SELECT
        c.customer_key,
        c.customer_number,
        c.first_name,
        c.last_name,
        COALESCE(c.country, 'Unknown') AS country,
        COUNT(DISTINCT f.order_number) AS total_orders,
        SUM(f.sales_amount) AS total_revenue
    FROM datawarehouseanalytics_gold.fact_sales f
    LEFT JOIN datawarehouseanalytics_gold.dim_customers c
        ON f.customer_key = c.customer_key
    GROUP BY
        c.customer_key,
        c.customer_number,
        c.first_name,
        c.last_name,
        COALESCE(c.country, 'Unknown')
),

ranked_customers AS (
    SELECT
        customer_key,
        customer_number,
        first_name,
        last_name,
        country,
        total_orders,
        total_revenue,
        DENSE_RANK() OVER (
            PARTITION BY country
            ORDER BY total_revenue DESC
        ) AS country_customer_rank
    FROM customer_sales
)

SELECT *
FROM ranked_customers
WHERE country_customer_rank <= 3
ORDER BY country, country_customer_rank;

In [0]:
%sql
/*
Category Revenue Contribution

Purpose:
    Rank categories by revenue and calculate cumulative revenue contribution.
    This helps identify whether revenue is concentrated in a few categories.
*/

WITH category_sales AS (
    SELECT
        COALESCE(p.category, 'Unknown') AS category,
        SUM(f.sales_amount) AS total_revenue
    FROM datawarehouseanalytics_gold.fact_sales f
    LEFT JOIN datawarehouseanalytics_gold.dim_products p
        ON f.product_key = p.product_key
    GROUP BY COALESCE(p.category, 'Unknown')
),

ranked_categories AS (
    SELECT
        category,
        total_revenue,
        ROUND(
            total_revenue * 100.0 / SUM(total_revenue) OVER (),
            2
        ) AS revenue_percentage,
        DENSE_RANK() OVER (ORDER BY total_revenue DESC) AS category_rank
    FROM category_sales
)

SELECT
    category,
    total_revenue,
    revenue_percentage,
    ROUND(
        SUM(revenue_percentage) OVER (
            ORDER BY category_rank
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ),
        2
    ) AS cumulative_revenue_percentage,
    category_rank
FROM ranked_categories
ORDER BY category_rank;